In [9]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [10]:
import os
import sys
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, "..", "..", ".."))
sys.path.append(project_root)
from src.carburants.fuel_price_utils import get_watermark, update_watermark, merge_dimension, full_keys


## Lecture table contexte et table silver

In [11]:
TABLE_DE_CONTEXT = "fuel_price_dev.silver.update_silver_context"

# Watermark = dernière valeur déjà traitée, mémorisée dans une table de contrôle
# -> on ne relit que le delta bronze 
wm_dim_carburant = get_watermark(spark, TABLE_DE_CONTEXT, "dim_carburant")
wm_dim_geo       = get_watermark(spark, TABLE_DE_CONTEXT, "dim_geo")
wm_dim_station   = get_watermark(spark, TABLE_DE_CONTEXT, "dim_station")
wm_fait_prix     = get_watermark(spark, TABLE_DE_CONTEXT, "fait_prix")
wm_fait_rupture  = get_watermark(spark, TABLE_DE_CONTEXT, "fait_rupture")

brze_dim_carburant_df = spark.read.table("fuel_price_dev.bronze.brze_dim_carburant")
brze_dim_geo_df       = spark.read.table("fuel_price_dev.bronze.brze_dim_geo")
brze_dim_station_df   = spark.read.table("fuel_price_dev.bronze.brze_dim_station")
brze_fait_prix_df     = spark.read.table("fuel_price_dev.bronze.brze_fait_prix")
brze_fait_rupture_df  = spark.read.table("fuel_price_dev.bronze.brze_fait_rupture")

## Comparaison entre les lignes Bronze et la date_maj de la table update_silver_context

In [12]:
if wm_dim_carburant is not None:
    brze_dim_carburant_df = brze_dim_carburant_df.filter(col("date_ingestion") > wm_dim_carburant)
if wm_dim_geo is not None:
    brze_dim_geo_df = brze_dim_geo_df.filter(col("date_ingestion") > wm_dim_geo)
if wm_dim_station is not None:
    brze_dim_station_df = brze_dim_station_df.filter(col("date_ingestion") > wm_dim_station)
if wm_fait_prix is not None:
    brze_fait_prix_df = brze_fait_prix_df.filter(col("date_ingestion") > wm_fait_prix)
if wm_fait_rupture is not None:
    brze_fait_rupture_df = brze_fait_rupture_df.filter(col("date_ingestion") > wm_fait_rupture)

## Mise à jour des tables silver si données plus récentes

In [13]:
slv_dim_carburant_df = brze_dim_carburant_df\
            .withColumn("date_traitement", current_timestamp())\
            .withColumn("id", col("id").cast(IntegerType()))\
            .filter(col("id").isNotNull())\
            .select("id","nom","date_ingestion", "date_traitement")\
            .dropDuplicates(["id"])

slv_dim_geo_df = brze_dim_geo_df\
            .withColumn("date_traitement", current_timestamp())\
            .filter(col("code_region").isNotNull() & col("code_departement").isNotNull() )\
            .select(
                "code_region",
                "region",
                "code_departement",
                "departement",
                "date_ingestion",
                "date_traitement"
        ).dropDuplicates(["code_departement",])


dim_geo_keys_df = full_keys(spark, "fuel_price_dev.silver.dim_geo", slv_dim_geo_df, ["code_departement"])

slv_dim_station_df = brze_dim_station_df\
                      .withColumn("date_traitement", current_timestamp())\
                      .filter(col("station_id").isNotNull())\
                      .dropDuplicates(["station_id"])\
                      .select(
                          "station_id",
                          "adresse",
                          "ville",
                          "code_postal",
                          "code_departement",
                          "service",
                          "code_region",
                          "date_ingestion",
                          "date_traitement"
                )\
             .join(
                dim_geo_keys_df,
                on="code_departement",
                how="left_semi"
                )

# Clés carburant/station valides = table silver déjà mergée UNION delta du run courant
dim_carburant_keys_df = full_keys(spark, "fuel_price_dev.silver.dim_carburant", slv_dim_carburant_df, ["id"])
dim_station_keys_df   = full_keys(spark, "fuel_price_dev.silver.dim_station", slv_dim_station_df, ["station_id"])

In [ ]:
# upsert sur la clé métier pour éviter les doublons à chaque snapshot bronze
merge_dimension(spark, slv_dim_carburant_df, "fuel_price_dev.silver.dim_carburant", ["id"])
merge_dimension(spark, slv_dim_geo_df, "fuel_price_dev.silver.dim_geo", ["code_departement"])
merge_dimension(spark, slv_dim_station_df, "fuel_price_dev.silver.dim_station", ["station_id"])

In [15]:
# ============================================================================
# BRONZE FACTS  TRANSFORMATIONS
# ============================================================================
slv_fait_prix_df = (
    brze_fait_prix_df
    .withColumn("date_traitement", current_timestamp())
    .withColumn("carburant_id", col("carburant_id").cast(IntegerType()))
    .withColumn("date_maj", expr("try_cast(date_maj AS TIMESTAMP)"))
    .withColumn("prix", col("prix").cast(DoubleType()))
    .filter(col("prix").isNotNull() & col("carburant_id").isNotNull())
    .select(
        "id_fct_pr",
        "station_id",
        "carburant_id",
        "date_maj",
        "prix", 
        "date_ingestion",
        "date_traitement"
    )
    .join(
        dim_carburant_keys_df.select(col("id").alias("carburant_id")),
        on="carburant_id",
        how="left_semi"
    )
    .join(
        dim_station_keys_df,
        on="station_id",
        how="left_semi"
    )
    .dropDuplicates(["id_fct_pr", "date_maj"])
)


slv_fait_rupture_df = (
    brze_fait_rupture_df
    .withColumn("date_traitement", current_timestamp())
    .withColumn("id_carburant", col("id_carburant").cast(IntegerType()))
    .withColumn("debut_rupture", expr("try_cast(debut_rupture AS TIMESTAMP)"))
    .withColumn("fin_rupture", expr("try_cast(fin_rupture AS TIMESTAMP)"))
    .filter(col("id_carburant").isNotNull() & col("debut_rupture").isNotNull())
    .select(
        "id_fct_rpt",
        "station_id",
        "id_carburant",
        "debut_rupture",
        "fin_rupture",
        "type_rupture",
        "date_ingestion", 
        "date_traitement"
    )
    .join(
        dim_carburant_keys_df.select(col("id").alias("id_carburant")),
        on="id_carburant",
        how="left_semi"
    )
    .join(
        dim_station_keys_df,
        on="station_id",
        how="left_semi"
    )
    .dropDuplicates(["id_fct_rpt", "debut_rupture", "fin_rupture"])
)

In [ ]:
slv_fait_prix_df.write\
                    .format("delta")\
                    .mode("append")\
                    .saveAsTable("fuel_price_dev.silver.fait_prix")

slv_fait_rupture_df.write\
                    .format("delta")\
                    .mode("append")\
                    .saveAsTable("fuel_price_dev.silver.fait_rupture")

## Update update_silver_context

In [17]:
# Update de la table de contexte si le slv_dim contient des données
# Update avec la date_ingestion la plus recente  pour chaque table du silver
update_watermark(spark, TABLE_DE_CONTEXT, "dim_carburant", slv_dim_carburant_df.agg(max("date_ingestion")).collect()[0][0])
update_watermark(spark, TABLE_DE_CONTEXT, "dim_geo", slv_dim_geo_df.agg(max("date_ingestion")).collect()[0][0])
update_watermark(spark, TABLE_DE_CONTEXT, "dim_station", slv_dim_station_df.agg(max("date_ingestion")).collect()[0][0])
update_watermark(spark, TABLE_DE_CONTEXT, "fait_prix", slv_fait_prix_df.agg(max("date_ingestion")).collect()[0][0])
update_watermark(spark, TABLE_DE_CONTEXT, "fait_rupture", slv_fait_rupture_df.agg(max("date_ingestion")).collect()[0][0])